# Lesson 2.2 — CloudWatch Logs, Log Insights & CloudWatch Agent

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Create CloudWatch Log Groups, Log Streams, and configure retention policies
- Ingest logs from applications using boto3
- Write CloudWatch Logs Insights queries to analyze log data
- Understand and configure the CloudWatch Agent on EC2
- Create Metric Filters to convert log patterns into CloudWatch metrics

**Exam relevance:** SOA-C03 Domain 1, Skills 1.1.1, 1.1.2

_Reference: [CloudWatch Logs Guide](https://docs.aws.amazon.com/cloudwatch/latest/logs/WhatIsCloudWatchLogs.html) | [Log Insights Query Syntax](https://docs.aws.amazon.com/cloudwatch/latest/logs/CWL_QuerySyntax.html)_

## 1. CloudWatch Logs Concepts

CloudWatch Logs is a **managed log storage and analysis** service. The hierarchy is:

```
Log Group                     ← Container (e.g., /aws/lambda/my-function)
  └── Log Stream              ← Source of logs (e.g., 2024/01/01/[$LATEST]abc123)
        └── Log Events        ← Individual log lines with timestamps
```

### Log Group Naming Conventions
AWS services use predictable naming:
- `/aws/lambda/<function-name>` — Lambda logs
- `/aws/rds/instance/<db-id>/error` — RDS logs
- `/aws/ecs/cluster/<cluster>` — ECS container logs
- `/aws/cloudtrail/<account-id>` — CloudTrail API logs

### Retention Policies
By default, CloudWatch Logs **never expire** (cost increases indefinitely!). Always set retention:
- Valid values: 1, 3, 5, 7, 14, 30, 60, 90, 120, 150, 180, 365, 400, 545, 731, 1827, 3653 days
- Use **S3 export** for long-term archival (much cheaper than CloudWatch Logs)

### The CloudWatch Agent
The **CloudWatch Agent** runs on EC2 (or on-premises servers) to:
- Collect **OS-level metrics** not available by default (memory usage, disk I/O per path)
- Stream **application log files** to CloudWatch Logs
- Collect **custom metrics** from StatsD/collectd

The agent is configured via a JSON config file and uses an IAM role (`CloudWatchAgentServerPolicy`) for permissions.

In [ ]:
import boto3
import json
import time
from datetime import datetime, timezone

logs = boto3.client('logs', region_name='us-east-1')
REGION = 'us-east-1'
LOG_GROUP = '/cloudops-learning/demo-app'
LOG_STREAM = f"app-server-{datetime.now().strftime('%Y-%m-%d')}"

print(f"Log Group: {LOG_GROUP}")
print(f"Log Stream: {LOG_STREAM}")

In [ ]:
# Create a Log Group with retention policy
try:
    logs.create_log_group(LogGroupName=LOG_GROUP)
    print(f"Created log group: {LOG_GROUP}")
except logs.exceptions.ResourceAlreadyExistsException:
    print(f"Log group already exists: {LOG_GROUP}")

# Set retention policy — ALWAYS do this!
logs.put_retention_policy(LogGroupName=LOG_GROUP, retentionInDays=30)
print("Set retention: 30 days")

# Create a Log Stream
try:
    logs.create_log_stream(LogGroupName=LOG_GROUP, LogStreamName=LOG_STREAM)
    print(f"Created log stream: {LOG_STREAM}")
except logs.exceptions.ResourceAlreadyExistsException:
    print(f"Log stream already exists: {LOG_STREAM}")

In [ ]:
# Simulate ingesting application logs into CloudWatch
import random
from datetime import datetime, timezone

# Generate realistic log events
log_levels = ['INFO', 'INFO', 'INFO', 'WARN', 'ERROR']
endpoints = ['/api/orders', '/api/users', '/api/products', '/health']
user_ids = ['u-1001', 'u-1002', 'u-1003', 'u-9999']

log_events = []
base_time = int(datetime.now(timezone.utc).timestamp() * 1000)

for i in range(20):
    level = random.choice(log_levels)
    endpoint = random.choice(endpoints)
    latency = random.randint(20, 800)
    status = 200 if level == 'INFO' else (500 if level == 'ERROR' else 400)
    user = random.choice(user_ids)
    
    message = (
        f"{level} RequestId=req-{i:04d} "
        f"userId={user} endpoint={endpoint} "
        f"method=GET statusCode={status} "
        f"duration={latency}ms"
    )
    
    log_events.append({
        'timestamp': base_time + (i * 5000),  # 5-second intervals
        'message': message
    })

# Send log events to CloudWatch
response = logs.put_log_events(
    logGroupName=LOG_GROUP,
    logStreamName=LOG_STREAM,
    logEvents=log_events
)

print(f"Ingested {len(log_events)} log events.")
print("Sample log entries:")
for event in log_events[:5]:
    print(f"  {event['message']}")

In [ ]:
# CloudWatch Logs Insights — query and analyze logs
# Insights uses a SQL-like syntax optimized for log analysis
import time
from datetime import datetime, timedelta, timezone

logs = boto3.client('logs', region_name='us-east-1')

end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(hours=1)

# Query: Find all ERROR logs and calculate average latency by endpoint
query = """
fields @timestamp, @message
| parse @message "endpoint=* duration=*ms" as endpoint, duration
| filter @message like /ERROR/
| stats count() as errorCount, avg(duration) as avgLatency by endpoint
| sort errorCount desc
"""

# Start the query
response = logs.start_query(
    logGroupName=LOG_GROUP,
    startTime=int(start_time.timestamp()),
    endTime=int(end_time.timestamp()),
    queryString=query
)
query_id = response['queryId']
print(f"Query started: {query_id}")

# Poll until query completes
for _ in range(30):
    time.sleep(2)
    result = logs.get_query_results(queryId=query_id)
    if result['status'] == 'Complete':
        break

print(f"\nQuery Status: {result['status']}")
print(f"Scanned: {result['statistics']['recordsScanned']:.0f} records")
print("\nResults (Errors by Endpoint):")
for row in result.get('results', []):
    fields = {r['field']: r['value'] for r in row}
    print(f"  Endpoint: {fields.get('endpoint', 'N/A'):20} Errors: {fields.get('errorCount', 0):3} AvgLatency: {fields.get('avgLatency', 'N/A')}ms")

In [ ]:
# Create a Metric Filter — convert log patterns into CloudWatch metrics
# This is powerful: turn log data into actionable metrics WITHOUT code changes

logs.put_metric_filter(
    logGroupName=LOG_GROUP,
    filterName='ErrorRateFilter',
    filterPattern='[level="ERROR", ...]',    # Pattern: match log lines with ERROR
    metricTransformations=[
        {
            'metricName': 'ApplicationErrorCount',
            'metricNamespace': 'CloudOpsLearning/Demo',
            'metricValue': '1',                    # Count each matching event
            'unit': 'Count',
            'defaultValue': 0                      # Use 0 when no matches (prevents gaps)
        }
    ]
)

print("Metric filter 'ErrorRateFilter' created.")
print("CloudWatch will now count ERROR log lines as a CloudWatch metric.")
print("Namespace: CloudOpsLearning/Demo")
print("Metric: ApplicationErrorCount")

## CloudWatch Agent Configuration

The CloudWatch Agent config file controls what metrics and logs the agent collects. Here is a production-ready example for a Linux EC2 instance:

In [ ]:
%%writefile cloudwatch-agent-config.json
{
  "agent": {
    "metrics_collection_interval": 60,
    "run_as_user": "cwagent"
  },
  "metrics": {
    "namespace": "CloudOpsLearning/EC2",
    "append_dimensions": {
      "InstanceId": "${aws:InstanceId}",
      "AutoScalingGroupName": "${aws:AutoScalingGroupName}"
    },
    "metrics_collected": {
      "mem": {
        "measurement": ["mem_used_percent"],
        "metrics_collection_interval": 60
      },
      "disk": {
        "measurement": ["used_percent"],
        "resources": ["/", "/var"],
        "metrics_collection_interval": 60
      },
      "cpu": {
        "measurement": ["cpu_usage_idle", "cpu_usage_user", "cpu_usage_system"],
        "metrics_collection_interval": 60,
        "totalcpu": true
      },
      "net": {
        "measurement": ["bytes_sent", "bytes_recv", "packets_sent", "packets_recv"],
        "resources": ["eth0"],
        "metrics_collection_interval": 60
      }
    }
  },
  "logs": {
    "logs_collected": {
      "files": {
        "collect_list": [
          {
            "file_path": "/var/log/messages",
            "log_group_name": "/cloudops-learning/ec2/system",
            "log_stream_name": "{instance_id}/system",
            "retention_in_days": 30
          },
          {
            "file_path": "/var/log/app/application.log",
            "log_group_name": "/cloudops-learning/demo-app",
            "log_stream_name": "{instance_id}/app",
            "multi_line_start_pattern": "^\\d{4}-\\d{2}-\\d{2}",
            "retention_in_days": 30
          }
        ]
      }
    }
  }
}

### Installing the CloudWatch Agent on EC2

On a new EC2 instance with the EC2CloudWatchRole (created in Lesson 1.1), run:

```bash
# Download the agent
sudo yum install -y amazon-cloudwatch-agent

# Upload config to SSM Parameter Store (then agent can pull it)
aws ssm put-parameter \
  --name /cloudwatch-agent/config/demo \
  --type String \
  --value file://cloudwatch-agent-config.json

# Start agent with config from SSM
sudo /opt/aws/amazon-cloudwatch-agent/bin/amazon-cloudwatch-agent-ctl \
  -a fetch-config \
  -m ec2 \
  -s \
  -c ssm:/cloudwatch-agent/config/demo

# Verify agent is running
sudo /opt/aws/amazon-cloudwatch-agent/bin/amazon-cloudwatch-agent-ctl -m ec2 -a status
```

**Key metrics the agent adds (not available by default):**
- `mem_used_percent` — Memory utilization (EC2 default metrics have NO memory!)
- `disk_used_percent` — Disk utilization per mount point
- `cpu_usage_idle` — Per-core CPU breakdown

## Exercise

In [ ]:
# TODO: Write a CloudWatch Logs Insights query that:
# 1. Searches the LOG_GROUP
# 2. Parses the 'duration' field from log messages
# 3. Calculates the p99 (99th percentile) latency per endpoint
# 4. Returns results sorted by p99 descending
# Hint: Logs Insights has a `pct(@duration_field, 99)` function

import boto3, time
from datetime import datetime, timedelta, timezone

logs = boto3.client('logs', region_name='us-east-1')
LOG_GROUP = '/cloudops-learning/demo-app'

query = """
# TODO: Write your Logs Insights query here
fields @timestamp, @message
| limit 5
"""

end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(hours=1)

response = logs.start_query(
    logGroupName=LOG_GROUP,
    startTime=int(start_time.timestamp()),
    endTime=int(end_time.timestamp()),
    queryString=query
)

# Poll for results
for _ in range(15):
    time.sleep(2)
    result = logs.get_query_results(queryId=response['queryId'])
    if result['status'] == 'Complete':
        break

print(f"Status: {result['status']}")
for row in result.get('results', []):
    print({r['field']: r['value'] for r in row})

In [ ]:
# Cleanup
import boto3, os

logs = boto3.client('logs', region_name='us-east-1')
LOG_GROUP = '/cloudops-learning/demo-app'

try:
    logs.delete_metric_filter(logGroupName=LOG_GROUP, filterName='ErrorRateFilter')
    print("Deleted metric filter")
except Exception as e:
    print(f"Metric filter: {e}")

try:
    logs.delete_log_group(logGroupName=LOG_GROUP)
    print(f"Deleted log group: {LOG_GROUP}")
except Exception as e:
    print(f"Log group: {e}")

if os.path.exists('cloudwatch-agent-config.json'):
    os.remove('cloudwatch-agent-config.json')

## Summary

- CloudWatch Logs hierarchy: **Log Group → Log Stream → Log Events**
- **Always set retention policies** — logs never expire by default
- **Logs Insights** is a powerful query engine; learn `parse`, `filter`, `stats`, `sort`
- **Metric Filters** convert log patterns into CloudWatch metrics — zero code changes needed
- **CloudWatch Agent** adds memory, disk, and application log collection to EC2 instances
- Store the agent config in **SSM Parameter Store** for centralized management

## Key Exam Points (SOA-C03)
- EC2 default CloudWatch metrics do **NOT** include memory or disk — need the CloudWatch Agent
- `multi_line_start_pattern` in agent config handles stack traces
- Logs Insights is **not real-time** — runs asynchronously (use `start_query` + `get_query_results`)

---

**Next:** [Lesson 2.3 — CloudTrail, EventBridge & SNS Notifications](lesson-2.3-cloudtrail-eventbridge.ipynb)